# Single-Contract Model Runs (AG)

This notebook trains four models (Logistic Regression, SVM, Random Forest, XGBoost if available) on the AG contract using the shared Python modules in `esl_project`. Each model is trained in its own cell, followed by a visualization cell. Figures are saved with timestamped names into per-contract folders.


In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from esl_project.data_utils import DataLoadConfig, list_csv_files
from esl_project.pipelines import ContractRunConfig, run_single_contract, short_contract_tag

plt.style.use("seaborn-v0_8-whitegrid")

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("outputs_parallel") / f"ESL_run_{RUN_TIMESTAMP}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("2005年__20250905")
print(f"Run timestamp: {RUN_TIMESTAMP}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")

all_files = list_csv_files(DATA_DIR)
ag_files = [p for p in all_files if p.name.startswith("AG_")]
if not ag_files:
    raise FileNotFoundError("AG contract CSV not found in data directory.")
AG_PATH = ag_files[0]
TAG = short_contract_tag(AG_PATH.stem)
print(f"Using contract file: {AG_PATH.name} (tag={TAG})")

load_cfg = DataLoadConfig(
    data_dir=DATA_DIR,
    max_files=1,
    nrows_per_file=None,
    start_date="2014-01-01",
    end_date="2020-12-31",
)


In [ ]:
all_files = list_csv_files(DATA_DIR)
ag_files = [p for p in all_files if p.name.startswith("AG_")]
if not ag_files:
    raise FileNotFoundError("AG contract CSV not found in data directory.")
AG_PATH = ag_files[0]
TAG = short_contract_tag(AG_PATH.stem)
print(f"Using contract file: {AG_PATH.name} (tag={TAG})")

load_cfg = DataLoadConfig(
    data_dir=DATA_DIR,
    max_files=1,
    nrows_per_file=None,
    start_date="2014-01-01",
    end_date="2020-12-31",
)


In [ ]:
def plot_confusion_and_rolling(model_label: str, tag: str, results: dict):
    """Plot confusion heatmap and rolling metrics, then save with timestamped names."""
    fig_dir = results["fig_dir"]
    fig_dir.mkdir(parents=True, exist_ok=True)

    conf = pd.DataFrame(results["confusion"], index=[-1, 0, 1], columns=[-1, 0, 1])
    metrics_df = results["rolling_metrics"]

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(conf, annot=True, fmt="g", cmap="Blues", ax=ax)
    ax.set_title(f"{tag} {model_label} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.savefig(fig_dir / f"{tag}_{model_label}_confusion_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    if not metrics_df.empty:
        ax.plot(metrics_df["test_end"], metrics_df["accuracy"], marker="o", label="Accuracy")
        ax.plot(metrics_df["test_end"], metrics_df["f1_macro"], marker="o", label="Macro F1")
        ax.tick_params(axis="x", labelrotation=45)
        fig.autofmt_xdate()
    ax.set_title(f"{tag} {model_label} Rolling Performance")
    ax.set_xlabel("Test month end")
    ax.set_ylabel("Score")
    ax.legend()
    fig.savefig(fig_dir / f"{tag}_{model_label}_rolling_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
logit_cfg = ContractRunConfig(
    candidate_C=[0.01, 0.1],
    model_name="logit",
    train_months=12,
    test_months=1,
    downsample_every=2,
)
logit_results = run_single_contract(AG_PATH, OUTPUT_ROOT, logit_cfg, load_cfg)
print("[Logit] Best params:", logit_results.get("best_params"))
print("[Logit] Summary head:
", logit_results.get("summary").head())


In [ ]:
plot_confusion_and_rolling("logit", TAG, logit_results)


In [ ]:
svm_cfg = ContractRunConfig(
    candidate_C=[0.5],
    model_name="svm",
    param_grid=[{"C": 0.5, "gamma": "scale"}],
    train_months=12,
    test_months=1,
    downsample_every=5,
)
svm_results = run_single_contract(AG_PATH, OUTPUT_ROOT, svm_cfg, load_cfg)
print("[SVM] Best params:", svm_results.get("best_params"))
print("[SVM] Summary head:
", svm_results.get("summary").head())


In [ ]:
plot_confusion_and_rolling("svm", TAG, svm_results)


In [ ]:
rf_cfg = ContractRunConfig(
    candidate_C=[0.1],
    model_name="rf",
    param_grid=[{"n_estimators": 120, "max_depth": 8, "max_features": "sqrt"}],
    train_months=12,
    test_months=1,
    downsample_every=5,
)
rf_results = run_single_contract(AG_PATH, OUTPUT_ROOT, rf_cfg, load_cfg)
print("[RF] Best params:", rf_results.get("best_params"))
print("[RF] Summary head:
", rf_results.get("summary").head())


In [ ]:
plot_confusion_and_rolling("rf", TAG, rf_results)


In [ ]:
xgb_available = importlib.util.find_spec("xgboost") is not None
if not xgb_available:
    print("[XGB] xgboost is not installed; skipping this run.")
    xgb_results = None
else:
    # Patch train_model so XGB sees labels 0/1/2 but returns predictions mapped back to -1/0/1
    import numpy as np
    import esl_project.model_loader as ml
    from esl_project.models.xgboost_model import train_xgboost as _train_xgb

    _orig_train_model = ml.train_model

    class XGBLabelWrapper:
        def __init__(self, base_model, inv_map):
            self.model = base_model
            self.inv_map = inv_map
            self._order = [-1, 0, 1]

        def predict(self, X):
            raw = self.model.predict(X)
            return np.array([self.inv_map[int(v)] for v in raw])

        def predict_proba(self, X):
            base = self.model.predict_proba(X)
            out = np.zeros((base.shape[0], len(self._order)))
            mapping = {0: -1, 1: 0, 2: 1}
            for src_lbl, tgt_lbl in mapping.items():
                tgt_idx = self._order.index(tgt_lbl)
                out[:, tgt_idx] = base[:, src_lbl]
            return out

        def __getattr__(self, name):
            return getattr(self.model, name)

    def patched_train_model(X_train, y_train, config):
        if config.model_name.lower() == "xgb":
            label_map = {-1: 0, 0: 1, 1: 2}
            inv_map = {v: k for k, v in label_map.items()}
            y_shift = np.array([label_map[int(v)] for v in y_train])
            base = _train_xgb(X_train, y_shift, **config.params)
            return XGBLabelWrapper(base, inv_map)
        return _orig_train_model(X_train, y_train, config)

    ml.train_model = patched_train_model

    xgb_cfg = ContractRunConfig(
        candidate_C=[0.1],
        model_name="xgb",
        param_grid=[{"n_estimators": 120, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.8, "colsample_bytree": 0.8}],
        train_months=12,
        test_months=1,
        downsample_every=5,
    )
    xgb_results = run_single_contract(AG_PATH, OUTPUT_ROOT, xgb_cfg, load_cfg)
    print("[XGB] Best params:", xgb_results.get("best_params"))
    print("[XGB] Summary head:", xgb_results.get("summary").head())


In [ ]:
if xgb_results is not None:
    plot_confusion_and_rolling("xgb", TAG, xgb_results)
else:
    print("[XGB] Visualization skipped because training did not run.")
